# Practical File 1 — Image Acquisition and Representation (Lab 1 & 2)

**Based on the 10 questions on slide 62 of the supplied PPT.**

## How to run
1. Keep this notebook and `image_sources.md` in the same folder.
2. Run the cells from top to bottom. The first code cell downloads a different set of real photographs for `flower.jpg`, `scene.jpg`, `sunset.png`, `car.jpg`, and `sample.jpg`. The subjects remain the same, but the photographs have been changed.
3. An internet connection is needed on the first run. The new set uses its own marker so earlier photo files are replaced automatically.
4. Install dependencies if needed: `pip install opencv-python pillow matplotlib numpy`.

OpenCV reads colour images in **BGR** channel order. Convert BGR to RGB before displaying with Matplotlib. Photo source pages are in `image_sources.md`.


In [ ]:
from pathlib import Path
from io import BytesIO
from urllib.request import Request, urlopen
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

# Download a new set of real photographs on first run; subjects remain the same but photo files are different.
# The photo source pages are documented in image_sources.md.
PHOTO_URLS = {
    "flower.jpg": "https://images.unsplash.com/photo-1655811083761-4cc3a58a655e?auto=format&fit=crop&fm=jpg&q=85&w=1400",
    "scene.jpg": "https://images.unsplash.com/photo-1757465158284-ddaf4c1657c9?auto=format&fit=crop&fm=jpg&q=85&w=1400",
    "sunset.png": "https://images.unsplash.com/photo-1616930174006-09f7aa039863?auto=format&fit=crop&fm=jpg&q=85&w=1400",
    "car.jpg": "https://images.unsplash.com/photo-1556213313-fc5be959c4a9?auto=format&fit=crop&fm=jpg&q=85&w=1400",
    "sample.jpg": "https://images.unsplash.com/photo-1666008490793-338bc347ca2e?auto=format&fit=crop&fm=jpg&q=85&w=1400",
}

ready_marker = Path(".real_photos_different_subject_matched_set_ready")
expected_files = [*PHOTO_URLS.keys(), "img1.jpg", "img2.jpg"]
needs_download = (not ready_marker.exists()) or any(not Path(name).exists() for name in expected_files)

if needs_download:
    try:
        for filename, url in PHOTO_URLS.items():
            request = Request(url, headers={"User-Agent": "Mozilla/5.0 (Educational Image Processing Assignment)"})
            with urlopen(request, timeout=45) as response:
                raw = response.read()
            # Decode and re-save locally to ensure the extension matches the file format.
            with Image.open(BytesIO(raw)) as photo:
                photo = photo.convert("RGB")
                temp_path = Path(filename + ".part")
                if Path(filename).suffix.lower() == ".png":
                    photo.save(temp_path, format="PNG", optimize=True)
                else:
                    photo.save(temp_path, format="JPEG", quality=92, optimize=True)
            temp_path.replace(filename)
            print(f"Downloaded real photograph: {filename} ({photo.width} × {photo.height})")

        # Make an image pair from the same real photo at two JPEG quality levels
        # for the MSE exercise. These are processed copies, not artificial drawings.
        with Image.open("sample.jpg") as photo:
            photo = photo.convert("RGB")
            photo.save("img1.jpg", format="JPEG", quality=95, optimize=True)
            photo.save("img2.jpg", format="JPEG", quality=35, optimize=True)
        ready_marker.write_text("Different real photo set downloaded successfully.\n", encoding="utf-8")
    except Exception as exc:
        raise RuntimeError(
            "The real photographs could not be downloaded. Connect to the internet and rerun this cell. "
            "The image URLs and photo credits are listed in image_sources.md."
        ) from exc
else:
    print("Different real photographs already exist; using the local copies.")

print("Available photo files:", ", ".join(expected_files))


## Program 1 — Read `flower.jpg` with OpenCV and display it as “Original Image”

In [ ]:
img = cv2.imread("flower.jpg")
if img is None:
    raise FileNotFoundError("Could not read flower.jpg. Put it beside this notebook.")

# Notebook-friendly display; the displayed title is 'Original Image'.
plt.figure(figsize=(7, 5))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Original Image")
plt.axis("off")
plt.show()

# To open a separate desktop window when running as a normal local Python script:
# cv2.imshow("Original Image", img)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

## Program 2 — Print height, width, channels, and data type of `scene.jpg`

In [ ]:
img = cv2.imread("scene.jpg")
if img is None:
    raise FileNotFoundError("Could not read scene.jpg.")
height, width = img.shape[:2]
channels = 1 if img.ndim == 2 else img.shape[2]
print(f"Height: {height} pixels")
print(f"Width: {width} pixels")
print(f"Channels: {channels}")
print(f"Data type: {img.dtype}")

## Program 3 — Convert a colour image to grayscale and save as `output_gray.jpg`

In [ ]:
img = cv2.imread("flower.jpg")
if img is None:
    raise FileNotFoundError("Could not read flower.jpg.")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
if not cv2.imwrite("output_gray.jpg", gray):
    raise IOError("Could not save output_gray.jpg.")
print("Saved: output_gray.jpg")
plt.figure(figsize=(6, 4))
plt.imshow(gray, cmap="gray", vmin=0, vmax=255)
plt.title("Grayscale Image")
plt.axis("off")
plt.show()

## Program 4 — Open `sunset.png` with Pillow and display it

In [ ]:
with Image.open("sunset.png") as sunset:
    print("Image size (width, height):", sunset.size)
    print("Image mode:", sunset.mode)
    display(sunset.copy())
    # sunset.show() opens an external image viewer when run on a desktop.

## Program 5 — Resize an image to 50% of its original width and height

In [ ]:
img = cv2.imread("sample.jpg")
if img is None:
    raise FileNotFoundError("Could not read sample.jpg.")
h, w = img.shape[:2]
resized = cv2.resize(img, (max(1, w // 2), max(1, h // 2)), interpolation=cv2.INTER_AREA)
print("Original dimensions (width, height):", (w, h))
print("Resized dimensions (width, height):", (resized.shape[1], resized.shape[0]))
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.title("Original"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)); plt.title("50% Size"); plt.axis("off")
plt.tight_layout(); plt.show()

## Program 6 — Crop the central 200 × 200 pixel area and display it with Matplotlib

In [ ]:
img = cv2.imread("sample.jpg")
if img is None:
    raise FileNotFoundError("Could not read sample.jpg.")
h, w = img.shape[:2]
if h < 200 or w < 200:
    raise ValueError(f"Image must be at least 200×200 pixels; got {w}×{h}.")
x1, y1 = (w - 200) // 2, (h - 200) // 2
crop = img[y1:y1 + 200, x1:x1 + 200]
print("Crop shape (height, width, channels):", crop.shape)
plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)); plt.title("Central 200×200 Crop"); plt.axis("off"); plt.show()

## Program 7 — Rotate an image 90 degrees clockwise using OpenCV

In [ ]:
img = cv2.imread("sample.jpg")
if img is None:
    raise FileNotFoundError("Could not read sample.jpg.")
rotated = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
print("Rotated shape:", rotated.shape)
plt.imshow(cv2.cvtColor(rotated, cv2.COLOR_BGR2RGB)); plt.title("Rotated 90° Clockwise"); plt.axis("off"); plt.show()

## Program 8 — Flip `car.jpg` left-to-right (horizontal flip) and display it

In [ ]:
img = cv2.imread("car.jpg")
if img is None:
    raise FileNotFoundError("Could not read car.jpg.")
flipped = cv2.flip(img, 1)  # 1 means horizontal / left-to-right flip
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.title("Original Car"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(cv2.cvtColor(flipped, cv2.COLOR_BGR2RGB)); plt.title("Horizontal Flip"); plt.axis("off")
plt.tight_layout(); plt.show()

## Program 9 — Convert a colour image to HSV and display Hue, Saturation, and Value separately

In [ ]:
img = cv2.imread("sample.jpg")
if img is None:
    raise FileNotFoundError("Could not read sample.jpg.")
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
hue, saturation, value = cv2.split(hsv)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, channel, title in zip(axes, [hue, saturation, value], ["Hue", "Saturation", "Value"]):
    ax.imshow(channel, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout(); plt.show()
print("OpenCV HSV ranges: H = 0–179; S = 0–255; V = 0–255 for 8-bit images.")

## Program 10 — Compute Mean Squared Error (MSE) between grayscale `img1.jpg` and `img2.jpg`

In [ ]:
img1 = cv2.imread("img1.jpg", cv2.IMREAD_GRAYSCALE)
img2 = cv2.imread("img2.jpg", cv2.IMREAD_GRAYSCALE)
if img1 is None or img2 is None:
    raise FileNotFoundError("Both img1.jpg and img2.jpg must be available.")
if img1.shape != img2.shape:
    raise ValueError(f"Images must have the same dimensions. Got {img1.shape} and {img2.shape}.")
diff = img1.astype(np.float64) - img2.astype(np.float64)
mse = np.mean(diff ** 2)
print(f"Mean Squared Error (MSE): {mse:.4f}")
print("MSE = 0 means the two grayscale images are identical pixel by pixel.")

## Conclusion

These programs demonstrate image loading and display, image dimensions and data types, grayscale conversion, resizing, cropping, rotation, flipping, HSV channel representation, and image comparison using MSE. The programs use real photographs downloaded from Unsplash; this version changes the photographs while keeping the subjects the same.